In [ ]:
import pandas as pd
import numpy as np
from statistics import mean
import os
import sys

# Ruta absoluta al directorio src
src_path = os.path.abspath("../src")
sys.path.append(src_path)

# Ahora puedes importar el módulo utils
from utils.exploration_utils import preprocess_data, evaluar_escenarios_trading, calcular_features_vela
from utils.exploration_utils import feature_engineering_rsi, sma_engineering_from_close


In [ ]:
def read_csv_files(folder_path):
    # List all CSV files in the folder
    all_files = os.listdir(folder_path)
    csv_files = [os.path.join(folder_path, file) for file in all_files if file.endswith('.csv')]
    
    # Read and concatenate all CSV files into a single DataFrame
    combined_df = pd.concat((pd.read_csv(f) for f in csv_files), ignore_index=True)
    
    return combined_df

In [ ]:
# Read Data:
folder_path = '../data/EURUSD/5mins'
df_EURUSD_5min = read_csv_files(folder_path)
df_EURUSD_5min = preprocess_data(df_EURUSD_5min)

In [ ]:
import numpy as np
from numpy.lib.stride_tricks import sliding_window_view

X = 24
# Create sliding window view of future close values
future_high_values = sliding_window_view(df_EURUSD_5min['high'].values, window_shape=X)
future_low_values = sliding_window_view(df_EURUSD_5min['low'].values, window_shape=X)
future_close_values = sliding_window_view(df_EURUSD_5min['close'].values, window_shape=X)

# Pad with NaN at the end since we can't look into the future for the last X-1 rows

padding = np.full((X-1, X), np.nan)
future_high_values = np.vstack([future_high_values, padding])
future_low_values = np.vstack([future_low_values, padding])
future_close_values = np.vstack([future_close_values, padding])

# Assign to DataFrame
df_EURUSD_5min['future_high_values'] = future_high_values.tolist()
df_EURUSD_5min['future_low_values'] = future_low_values.tolist()
df_EURUSD_5min['future_close_values'] = future_close_values.tolist()

# Mostrar las primeras filas para verificar
df_EURUSD_5min.head()

In [ ]:
df_EURUSD_5min[['compra_resultado', 'compra_velas', 'compra_pips',
        'venta_resultado', 'venta_velas', 'venta_pips']] = df_EURUSD_5min.apply(
    lambda row: evaluar_escenarios_trading(row, pips_objetivo=20, pips_stop=5),
    axis=1
)

In [ ]:
df_EURUSD_5min.head()

In [ ]:
df_EURUSD_5min.iloc[1]

In [ ]:
print(df_EURUSD_5min.iloc[1]['close'])
print("-------ESCENARIO COMPRA-------")

print(df_EURUSD_5min.iloc[1]['close']+20/10000)
print(df_EURUSD_5min.iloc[1]['close']-5/10000)

print("-------ESCENARIO VENTA-------")
print(df_EURUSD_5min.iloc[1]['close']-20/10000)
print(df_EURUSD_5min.iloc[1]['close']+5/10000)

print("-------VALORES FUTUROS-------")
print(df_EURUSD_5min.iloc[1]['future_high_values'])
print(df_EURUSD_5min.iloc[1]['future_low_values'])

In [ ]:
# Calculate counts
counts = df_EURUSD_5min[['compra_resultado','venta_resultado']].value_counts()

# Calculate percentages
percentages = df_EURUSD_5min[['compra_resultado','venta_resultado']].value_counts(normalize=True) * 100

# Combine counts and percentages
results = pd.DataFrame({
    'Count': counts,
    'Percentage': percentages.round(2)
})

print(results)

In [ ]:
df_EURUSD_5min

In [ ]:
df_EURUSD_5min.head()

In [ ]:
pd.set_option('display.max_columns', None)

In [ ]:
df_EURUSD_5min = sma_engineering_from_close(df_EURUSD_5min, [10,40,100])

In [ ]:
df_EURUSD_5min.columns

In [ ]:
df_EURUSD_5min["multiclass_target"] = np.where(
    (df_EURUSD_5min['compra_resultado'] == 'OBJETIVO_ALCANZADO') & (df_EURUSD_5min['venta_resultado'] != 'OBJETIVO_ALCANZADO'),
    'objetivo_compra_alcanzado',
    np.where(
        (df_EURUSD_5min['compra_resultado'] != 'OBJETIVO_ALCANZADO') & (df_EURUSD_5min['venta_resultado'] == 'OBJETIVO_ALCANZADO'),
        'objetivo_venta_alcanzado',
        'no_objetivo_alcanzado'
    )
)

In [ ]:
df_EURUSD_5min.multiclass_target.value_counts()/df_EURUSD_5min.shape[0]

In [ ]:
# Filtrar datos de entrenamiento y prueba por fecha
df_train = df_EURUSD_5min[df_EURUSD_5min['Date'] < "20230101"]
df_test = df_EURUSD_5min[df_EURUSD_5min['Date'] >= "20230101"]

In [ ]:
# Definir las características (X) y el target (y)
features = ['diff_SMA_10_SMA_40', 'ratio_SMA_10_SMA_40', 'ratio_SMA_40_SMA_10',
       'diff_SMA_10_SMA_100', 'ratio_SMA_10_SMA_100', 'ratio_SMA_100_SMA_10',
       'diff_SMA_40_SMA_100', 'ratio_SMA_40_SMA_100', 'ratio_SMA_100_SMA_40',
       'sma_order_score', 'slope_SMA_10', 'slope_SMA_40', 'slope_SMA_100',
       'diff_slope_SMA_10_slope_SMA_40', 'diff_slope_SMA_10_slope_SMA_100',
       'diff_slope_SMA_40_slope_SMA_100', 'norm_diff_SMA_10_SMA_100',
       'norm_diff_SMA_40_SMA_100', 'close_diff_SMA_10', 'close_diff_SMA_40',
       'close_diff_SMA_100', 'crossover', 'sma_spread', 'acc_SMA_10',
       'acc_SMA_40', 'acc_SMA_100', 'SMA_10_momentum', 'SMA_10_pct_momentum',
       'SMA_40_momentum', 'SMA_40_pct_momentum', 'SMA_100_momentum',
       'SMA_100_pct_momentum', 'SMA_10_slope_mean', 'SMA_40_slope_mean',
       'SMA_100_slope_mean']


X_train = df_train[features]
y_train = df_train['multiclass_target']

X_test = df_test[features]
y_test = df_test['multiclass_target']

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

In [ ]:
# Create and train the RandomForestClassifier
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1,
    class_weight='balanced'
)

# Train the model
rf_model.fit(X_train, y_train)

# Make predictions
y_pred = rf_model.predict(X_test)

# Print classification report
print("Classification Report:\n")
print(classification_report(y_test, y_pred))

In [ ]:
np.arange(0.1, 1.0, 0.05)

In [ ]:
y_pred_proba = rf_model.predict_proba(X_test)

In [ ]:
import matplotlib.pyplot as plt

# Create a DataFrame with probabilities and actual results
prob_df = pd.DataFrame({
    'prob_no_objetivo': y_pred_proba[:, 0],
    'prob_compra': y_pred_proba[:, 1],
    'prob_venta': y_pred_proba[:, 2],
    'actual_compra': df_test['compra_resultado'],
    'actual_venta': df_test['venta_resultado']
})

# Create probability thresholds
thresholds = np.arange(0.1, 1.0, 0.05)

# Initialize lists to store metrics
compra_metrics = []
venta_metrics = []


# Calculate metrics for each threshold
for threshold in thresholds:
    # For compra (buy) predictions
    compra_preds = (prob_df['prob_compra'] > threshold)
    compra_correct = ((compra_preds) & (prob_df['actual_compra'] == 'OBJETIVO_ALCANZADO')).sum()
    compra_stop = ((compra_preds) & (prob_df['actual_compra'] == 'STOP_ALCANZADO')).sum()
    compra_total = compra_preds.sum()
    compra_precision = compra_correct/compra_total if compra_total > 0 else 0
    compra_profit = (compra_correct * 20) - (compra_stop * 5)
    compra_metrics.append((threshold, compra_precision, compra_total, compra_profit))
    
    
    # For venta (sell) predictions
    venta_preds = (prob_df['prob_venta'] > threshold)
    venta_correct = ((venta_preds) & (prob_df['actual_venta'] == 'OBJETIVO_ALCANZADO')).sum()
    venta_stop = ((venta_preds) & (prob_df['actual_venta'] == 'STOP_ALCANZADO')).sum()
    venta_total = venta_preds.sum()
    venta_precision = venta_correct/venta_total if venta_total > 0 else 0
    venta_profit = (venta_correct * 20) - (venta_stop * 5)
    venta_metrics.append((threshold, venta_precision, venta_total, venta_profit))

# Convert metrics to DataFrames
compra_df = pd.DataFrame(compra_metrics, columns=['threshold', 'precision', 'total_trades', 'profit'])
venta_df = pd.DataFrame(venta_metrics, columns=['threshold', 'precision', 'total_trades', 'profit'])

# Format and display buy metrics
print("Buy Strategy Metrics:")
display(compra_df.style
    .format({
        'threshold': '{:.2f}',
        'precision': '{:.2%}',
        'total_trades': '{:,.0f}',
        'profit': '{:,.0f}'
    })
    .set_properties(**{'text-align': 'center'})
    .set_table_styles([
        {'selector': 'th', 'props': [('text-align', 'center'), ('font-weight', 'bold')]},
        {'selector': '', 'props': [('border', '1px solid #ddd')]}
    ])
    .background_gradient(subset=['profit'], cmap='RdYlGn'))

print("\nSell Strategy Metrics:")
display(venta_df.style
    .format({
        'threshold': '{:.2f}',
        'precision': '{:.2%}',
        'total_trades': '{:,.0f}',
        'profit': '{:,.0f}'
    })
    .set_properties(**{'text-align': 'center'})
    .set_table_styles([
        {'selector': 'th', 'props': [('text-align', 'center'), ('font-weight', 'bold')]},
        {'selector': '', 'props': [('border', '1px solid #ddd')]}
    ])
    .background_gradient(subset=['profit'], cmap='RdYlGn'))
# Create plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Plot compra metrics
ax1.plot([x[0] for x in compra_metrics], [x[1] for x in compra_metrics], 'b-', label='Precision')
ax1.set_xlabel('Threshold')
ax1.set_ylabel('Precision')
ax1_twin = ax1.twinx()
ax1_twin.plot([x[0] for x in compra_metrics], [x[3] for x in compra_metrics], 'g:', label='Profit')
ax1_twin.set_ylabel('Profit')
ax1.set_title('Buy Signals')
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax1_twin.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right')

# Plot venta metrics
ax2.plot([x[0] for x in venta_metrics], [x[1] for x in venta_metrics], 'b-', label='Precision')
ax2.set_xlabel('Threshold')
ax2.set_ylabel('Precision')
ax2_twin = ax2.twinx()
ax2_twin.plot([x[0] for x in venta_metrics], [x[3] for x in venta_metrics], 'g:', label='Profit')
ax2_twin.set_ylabel('Profit')
ax2.set_title('Sell Signals')
lines1, labels1 = ax2.get_legend_handles_labels()
lines2, labels2 = ax2_twin.get_legend_handles_labels()
ax2.legend(lines1 + lines2, labels1 + labels2, loc='upper right')

plt.tight_layout()
plt.show()

In [ ]:
# Calculate counts

results_df =  df_test[['compra_resultado','venta_resultado']].copy()
results_df['y_true'] = y_test
results_df['y_pred'] = y_pred

counts = results_df[['y_pred','compra_resultado','venta_resultado']].value_counts()

# Calculate percentages
percentages = results_df[['y_pred','compra_resultado','venta_resultado']].value_counts(normalize=True) * 100

# Combine counts and percentages
results = pd.DataFrame({
    'Count': counts,
    'Percentage': percentages.round(2)
})

print(results)

In [ ]:
results_df = X_test.copy()
results_df['y_true'] = y_test
results_df['y_pred'] = y_pred

# Falsos positivos
fp_df = results_df[(results_df['y_true'] == 0) & (results_df['y_pred'] == 1)]

# Verdaderos positivos
tp_df = results_df[(results_df['y_true'] == 1) & (results_df['y_pred'] == 1)]

print(f"\nNúmero de Falsos Positivos: {fp_df.shape[0]}")
print(f"Número de Verdaderos Positivos: {tp_df.shape[0]}")

In [ ]:
df_EURUSD_5min = feature_engineering_rsi(df_EURUSD_5min, close_col='close', period=14)
df_EURUSD_5min = calcular_features_vela(df_EURUSD_5min)
df_EURUSD_5min = sma_engineering_from_close(df_EURUSD_5min, [5,20,60])

In [ ]:
import matplotlib.pyplot as plt

# Primero asegurarnos que tenemos la columna Month
if 'Month' not in df_EURUSD_5min.columns:
    df_EURUSD_5min['Month'] = pd.to_datetime(df_EURUSD_5min['date']).dt.strftime('%Y-%m')

# Calcular el ratio por mes
monthly_ratios = (df_EURUSD_5min
                 .groupby('Month')['compra_resultado']
                 .value_counts(normalize=True)
                 .unstack()
                 .fillna(0))

# Visualizar los resultados
print("Distribución mensual de eventos:")
print(monthly_ratios)

# Opcional: Visualización con gráfico de barras apiladas
ax = monthly_ratios.plot(kind='bar', stacked=True, figsize=(15, 6))
plt.title('Distribución de Eventos por Mes')
plt.xlabel('Mes')
plt.ylabel('Proporción')
plt.xticks(rotation=45)
plt.legend(title='Evento', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

# Opcional: Exportar a CSV para análisis posterior
monthly_ratios.to_csv('../data/monthly_event_ratios.csv')

In [ ]:
import matplotlib.pyplot as plt

# Primero asegurarnos que tenemos la columna Month
if 'Month' not in df_EURUSD_5min.columns:
    df_EURUSD_5min['Month'] = pd.to_datetime(df_EURUSD_5min['date']).dt.strftime('%Y-%m')

# Calcular el ratio por mes
monthly_ratios = (df_EURUSD_5min
                 .groupby('Month')['venta_resultado']
                 .value_counts(normalize=True)
                 .unstack()
                 .fillna(0))

# Visualizar los resultados
print("Distribución mensual de eventos:")
print(monthly_ratios)

# Opcional: Visualización con gráfico de barras apiladas
ax = monthly_ratios.plot(kind='bar', stacked=True, figsize=(15, 6))
plt.title('Distribución de Eventos por Mes')
plt.xlabel('Mes')
plt.ylabel('Proporción')
plt.xticks(rotation=45)
plt.legend(title='Evento', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

# Opcional: Exportar a CSV para análisis posterior
monthly_ratios.to_csv('../data/monthly_event_ratios.csv')

In [ ]:
df_EURUSD_5min.to_parquet('../data/parquet/EURUSD/5mins/data.parquet')

In [ ]:

# Asumiendo que el archivo está en la carpeta data
df_EURUSD_5min = pd.read_parquet('../data/parquet/EURUSD/5mins/data.parquet')

# Mostrar las primeras filas para verificar
print(df_EURUSD_5min.head())

# Información sobre el DataFrame
print("\nInformación del DataFrame:")
print(df_EURUSD_5min.info())

In [ ]:
df_EURUSD_5min.venta_resultado.value_counts()/df_EURUSD_5min.shape[0]

In [ ]:
df_EURUSD_5min["target_venta"] = np.where(df_EURUSD_5min["venta_resultado"] == "OBJETIVO_ALCANZADO", 1, 0)
df_EURUSD_5min.target_venta.value_counts()/df_EURUSD_5min.shape[0]

In [ ]:
df_EURUSD_5min.columns

### Intento 1. Solo variables de tendencia (RSI, Candlestick Values, SMA)

In [ ]:
# Filtrar datos de entrenamiento y prueba por fecha
df_train = df_EURUSD_5min[df_EURUSD_5min['Date'] < "20230101"]
df_test = df_EURUSD_5min[df_EURUSD_5min['Date'] >= "20230101"]

In [ ]:
# Definir las características (X) y el target (y)
features = ['RSI', 'rsi_overbought', 'rsi_oversold',
       'dist_rsi_70', 'dist_rsi_30', 'dist_rsi_50', 'rsi_slope', 'rsi_slope_3',
       'rsi_cross_30_up', 'rsi_cross_70_down', 'body_size', 'upper_wick',
       'lower_wick', 'total_range', 'body_ratio', 'upper_wick_ratio',
       'lower_wick_ratio', 'SMA_5', 'SMA_20', 'SMA_60', 'diff_SMA_5_SMA_20',
       'diff_SMA_5_SMA_60', 'diff_SMA_20_SMA_60', 'ratio_SMA_5_SMA_20',
       'ratio_SMA_20_SMA_5', 'ratio_SMA_5_SMA_60', 'ratio_SMA_60_SMA_5',
       'ratio_SMA_20_SMA_60', 'ratio_SMA_60_SMA_20', 'sma_order_score',
       'slope_SMA_5', 'slope_SMA_20', 'slope_SMA_60',
       'diff_slope_SMA_5_slope_SMA_20', 'diff_slope_SMA_5_slope_SMA_60',
       'diff_slope_SMA_20_slope_SMA_60', 'norm_diff_SMA_5_SMA_60',
       'norm_diff_SMA_20_SMA_60']


X_train = df_train[features]
y_train = df_train['target_venta']

X_test = df_test[features]
y_test = df_test['target_venta']

In [ ]:
import lightgbm as lgb
from sklearn.metrics import classification_report

In [ ]:
# Crear Datasets de LightGBM
lgb_train = lgb.Dataset(X_train, label=y_train)
lgb_test = lgb.Dataset(X_test, label=y_test, reference=lgb_train)



In [ ]:
# Calcular el peso para balancear las clases
def calculate_scale_pos_weight(y):
    """
    Calcula el scale_pos_weight basado en el desbalance de clases
    """
    neg_samples = (y == 0).sum()
    pos_samples = (y == 1).sum()
    return neg_samples / pos_samples

scale_pos_weight = calculate_scale_pos_weight(y_train)

In [ ]:
# Definir parámetros. LightGBM detectará columnas categóricas automáticamente si su dtype es category.
params = {
    'objective': 'binary',
    'metric': 'binary_logloss',
    'verbosity': -1,
    'seed': 42,
    'scale_pos_weight': scale_pos_weight
}

# Entrenar el modelo
model = lgb.train(
    params,
    lgb_train,
    valid_sets=[lgb_test]
)

In [ ]:
# Predecir
y_pred_proba = model.predict(X_test)  

In [ ]:
print(classification_report(y_test, y_pred_proba>0.5))

In [ ]:
results_df = X_test.copy()
results_df['y_true'] = y_test
results_df['y_pred'] = y_pred_proba>0.5

# Falsos positivos
fp_df = results_df[(results_df['y_true'] == 0) & (results_df['y_pred'] == 1)]

# Verdaderos positivos
tp_df = results_df[(results_df['y_true'] == 1) & (results_df['y_pred'] == 1)]

print(f"\nNúmero de Falsos Positivos: {fp_df.shape[0]}")
print(f"Número de Verdaderos Positivos: {tp_df.shape[0]}")

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

# Supongamos que tu DataFrame es df y ya contiene las 7 columnas especificadas.
# Ej: df = pd.read_csv("tus_datos.csv")

# Seleccionar las columnas de interés
features = ['body_size', 'upper_wick', 'lower_wick', 'total_range', 
            'body_ratio', 'upper_wick_ratio', 'lower_wick_ratio']
          

X = df_EURUSD_5min[features].copy()



In [ ]:
kmeans = KMeans(n_clusters=15, random_state=42)
kmeans.fit(X)

# Obtener etiquetas de clúster
cluster_labels = kmeans.labels_

# Añadirlas al DataFrame original
df_EURUSD_5min['cluster'] = cluster_labels

# Visualizar el resultado
df_EURUSD_5min.head()

In [ ]:
df_EURUSD_5min.cluster.value_counts()

In [ ]:
window=50
info_window = get_time_window(df_EURUSD_5min, 'cluster', window)

In [ ]:

from sklearn.decomposition import PCA

# Supongamos que 'windows' es el array obtenido con la función get_time_window.
# Por ejemplo, windows tiene forma (n_samples, n_features)

# 1. Ajustar PCA sin reducir (para obtener todas las componentes y ver la varianza explicada)
pca_full = PCA(n_components=info_window.shape[1])
pca_full.fit(info_window)
explained_variance = pca_full.explained_variance_ratio_
cumulative_variance = np.cumsum(explained_variance)

# 2. Graficar la varianza acumulada (scree plot)
plt.figure(figsize=(8, 5))
plt.plot(range(1, len(cumulative_variance) + 1), cumulative_variance, marker='o', linestyle='--')
plt.xlabel('Número de componentes')
plt.ylabel('Varianza acumulada')
plt.title('Varianza acumulada por número de componentes')
plt.axhline(y=0.80, color='r', linestyle='-')
plt.text(1, 0.82, '80% de varianza', color='r')
plt.grid(True)
plt.show()



In [ ]:
# 3. Determinar el número mínimo de componentes que explican al menos el 80% de la varianza
n_components_80 = np.argmax(cumulative_variance >= 0.80) + 1
print("El número mínimo de componentes para explicar al menos el 80% de la varianza es:", n_components_80)

In [ ]:
# 4. Ajustar el PCA reduciendo a ese número de componentes
pca = PCA(n_components=0.80)  # Esto automáticamente selecciona el número mínimo para alcanzar el 80%
windows_reduced = pca.fit_transform(info_window)
print("Número de componentes seleccionados:", pca.n_components_)

In [ ]:
n_windows = len(df_EURUSD_5min) - window  # cantidad de ventanas que generamos
n_components = pca.n_components_

In [ ]:
# Convertir windows_reduced en DataFrame con columnas identificadas
pca_columns = [f'pca_{i+1}' for i in range(n_components)]
df_windows = pd.DataFrame(windows_reduced, columns=pca_columns)
df_windows.head()

In [ ]:
df_windows.index = df_EURUSD_5min.index[window:]  # window es 20, así que el primer índice será el 20
df_windows.head()

In [ ]:
# Realizar el merge (concatenar columnas)
df_EURUSD_5min_pca = pd.concat([df_EURUSD_5min, df_windows], axis=1)

In [ ]:
# Filtrar datos de entrenamiento y prueba por fecha
df_train = df_EURUSD_5min_pca[df_EURUSD_5min_pca['Date'] < "20230101"]
df_test = df_EURUSD_5min_pca[df_EURUSD_5min_pca['Date'] >= "20230101"]

In [ ]:
# Definir las características (X) y el target (y)
features = ['RSI', 'rsi_overbought', 'rsi_oversold',
       'dist_rsi_70', 'dist_rsi_30', 'dist_rsi_50', 'rsi_slope', 'rsi_slope_3',
       'rsi_cross_30_up', 'rsi_cross_70_down', 'body_size', 'upper_wick',
       'lower_wick', 'total_range', 'body_ratio', 'upper_wick_ratio',
       'lower_wick_ratio', 'SMA_5', 'SMA_20', 'SMA_60', 'diff_SMA_5_SMA_20',
       'diff_SMA_5_SMA_60', 'diff_SMA_20_SMA_60', 'ratio_SMA_5_SMA_20',
       'ratio_SMA_20_SMA_5', 'ratio_SMA_5_SMA_60', 'ratio_SMA_60_SMA_5',
       'ratio_SMA_20_SMA_60', 'ratio_SMA_60_SMA_20', 'sma_order_score',
       'slope_SMA_5', 'slope_SMA_20', 'slope_SMA_60',
       'diff_slope_SMA_5_slope_SMA_20', 'diff_slope_SMA_5_slope_SMA_60',
       'diff_slope_SMA_20_slope_SMA_60', 'norm_diff_SMA_5_SMA_60',
       'norm_diff_SMA_20_SMA_60','cluster'] + pca_columns


X_train = df_train[features]
y_train = df_train['target_venta']

X_test = df_test[features]
y_test = df_test['target_venta']

In [ ]:
results_df = X_test.copy()
results_df['y_true'] = y_test
results_df['y_pred'] = y_pred

# Falsos positivos
fp_df = results_df[(results_df['y_true'] == 0) & (results_df['y_pred'] == 1)]

# Verdaderos positivos
tp_df = results_df[(results_df['y_true'] == 1) & (results_df['y_pred'] == 1)]

print(f"\nNúmero de Falsos Positivos: {fp_df.shape[0]}")
print(f"Número de Verdaderos Positivos: {tp_df.shape[0]}")

In [ ]:
from optbinning import OptimalBinning

# Ajustar el binning óptimo sobre las probabilidades
# max_n_bins=5 para intentar obtener 5 segmentos
optb = OptimalBinning(name='y_pred_proba', dtype='numerical', solver='cp', max_n_bins=5)
optb.fit(y_pred_proba, y_test)

# Construir la tabla de binning
binning_table = optb.binning_table.build()

# binning_table es un objeto con métodos para explorar el resultado
# Para ver la tabla de resumen:
binning_table


In [ ]:
# Suponiendo que 'df' ya existe y tiene una columna 'cluster'

N = 4
# Generar las columnas desplazadas (lags) para las 10 velas anteriores
# cluster.shift(1) da el cluster de la vela previa, cluster.shift(2) de la vela ante-ante-última, etc.
lag_cols = [df_EURUSD_5min['cluster'].shift(i) for i in range(1, N+1)]

# Convertirlos en cadenas y concatenar
pattern_series = lag_cols[0].astype(str)
for i in range(1, N):
    pattern_series = pattern_series + "_" + lag_cols[i].astype(str)

# Agregar la nueva columna al DataFrame
df_EURUSD_5min['pattern'] = pattern_series


In [ ]:

# Visualizar el resultado
df_EURUSD_5min.pattern.value_counts()

In [ ]:
# Factorizar la columna pattern
df_EURUSD_5min['pattern_factorized'], uniques = pd.factorize(df_EURUSD_5min['pattern'])


In [ ]:
df_EURUSD_5min['pattern_factorized'].value_counts()

In [ ]:
# Creamos un DataFrame con toda la info necesaria
results_df = X_test.copy()
results_df['y_true'] = y_test
results_df['y_pred'] = y_pred

# Falsos positivos
fp_df = results_df[(results_df['y_true'] == 0) & (results_df['y_pred'] == 1)]

# Verdaderos positivos
tp_df = results_df[(results_df['y_true'] == 1) & (results_df['y_pred'] == 1)]

print(f"\nNúmero de Falsos Positivos: {fp_df.shape[0]}")
print(f"Número de Verdaderos Positivos: {tp_df.shape[0]}")

In [ ]:
df_EURUSD_5min.columns

In [ ]:
# Filtrar datos de entrenamiento y prueba por fecha
df_train = df_EURUSD_5min[df_EURUSD_5min['Date'] < "20230101"]
df_test = df_EURUSD_5min[df_EURUSD_5min['Date'] >= "20230101"]

In [ ]:
# Convertir la columna factorizada a tipo categoría para que LightGBM la reconozca como tal
df_train['pattern_factorized'] = df_train['pattern_factorized'].astype('category')
df_test['pattern_factorized'] = df_test['pattern_factorized'].astype('category')


In [ ]:
# Definir las características (X) y el target (y)
features = ['body_size', 'upper_wick', 'lower_wick', 'total_range', 
            'body_ratio', 'upper_wick_ratio', 'lower_wick_ratio', 'pattern_factorized','cluster','diff_SMA_5_SMA_20', 'diff_SMA_5_SMA_60', 'diff_SMA_20_SMA_60',
       'ratio_SMA_5_SMA_20', 'ratio_SMA_20_SMA_5', 'ratio_SMA_5_SMA_60',
       'ratio_SMA_60_SMA_5', 'ratio_SMA_20_SMA_60', 'ratio_SMA_60_SMA_20',
       'sma_order_score', 'slope_SMA_5', 'slope_SMA_20', 'slope_SMA_60',
       'diff_slope_SMA_5_slope_SMA_20', 'diff_slope_SMA_5_slope_SMA_60',
       'diff_slope_SMA_20_slope_SMA_60', 'norm_diff_SMA_5_SMA_60',
       'norm_diff_SMA_20_SMA_60']


X_train = df_train[features]
y_train = df_train['Target']

X_test = df_test[features]
y_test = df_test['Target']

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.optimizers import Adam


In [ ]:
# Seleccionar las columnas para el modelo
features = ['body_size', 'upper_wick', 'lower_wick', 'total_range', 
            'body_ratio', 'upper_wick_ratio', 'lower_wick_ratio', 'cluster']

# Convertir a numpy array
data = df[features].values

In [ ]:
months_SR = list(set(df_EURUSD_5min.loc[df_EURUSD_5min['Month'].astype(int)>201006,'Month'].astype(int).values))

In [ ]:
monthly_SR_levels = {}
for i in months_SR:
    data = df_EURUSD_5min[df_EURUSD_5min['Month'].astype(int)<i]
    
    resistance_array = cal_resistance_array(window=70, pips_range=0.0020,data=data['close'])[:15]
    support_array = cal_support_array(window=70, pips_range=0.0020,data=data['close'])[:15]

    resistance_levels = [x[0] for x in resistance_array]
    support_levels = [x[0] for x in support_array]

    # Store the results in the dictionary
    monthly_SR_levels[i] = {
        'resistance': resistance_levels,
        'support': support_levels
    }


In [ ]:
def calculate_distances_to_SR(candlestick_data, SR_levels):
    """Calculate the minimum distance of each candlestick to the nearest support and resistance levels."""
    resistances = SR_levels['resistance']
    supports = SR_levels['support']
    
    # Calculate distances for each candlestick in the DataFrame
    # Return Series of distances which will be added to the DataFrame
    min_distance_to_resistance = candlestick_data['close'].apply(
        lambda x: min(abs(x - r) for r in resistances))
    min_distance_to_support = candlestick_data['close'].apply(
        lambda x: min(abs(x - s) for s in supports))
    
    return min_distance_to_resistance, min_distance_to_support


In [ ]:
# Assuming df_EURUSD_5min is your main DataFrame and it contains a 'Month' column
# Iterate over each month in months_SR
for month in months_SR:
    month_str = str(month)
    
    # Filter data for the current month
    candlestick_data = df_EURUSD_5min[df_EURUSD_5min['Month'] == month_str]
    
    if month in monthly_SR_levels:
        # Calculate distances to S&R
        dist_to_res, dist_to_sup = calculate_distances_to_SR(candlestick_data, monthly_SR_levels[month])
        
        # Add the calculated distances back to the main DataFrame
        df_EURUSD_5min.loc[df_EURUSD_5min['Month'] == month_str, 'min_distance_to_resistance'] = dist_to_res
        df_EURUSD_5min.loc[df_EURUSD_5min['Month'] == month_str, 'min_distance_to_support'] = dist_to_sup
    else:
        print(f"No S&R data available for month {month_str}")

In [ ]:
df_EURUSD_5min.tail()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics  import classification_report, precision_score, recall_score
import xgboost as xgb

In [ ]:


# Ejemplo de uso:
# df = pd.read_csv("tus_datos.csv")  # con columna 'close'
# sma_periods = [5, 10, 20, 40, 60]
# df = feature_engineering_from_close(df, sma_periods)
# Ahora df tendrá SMA_5, SMA_10, ... y todas las características derivadas.


In [ ]:
import ta
# Varialbes de SMA
# Crear características técnicas simples
def create_technical_stats(df):
    #create SMA


    sma_cols = ['SMA_5', 'SMA_10', 'SMA_20', 'SMA_40', 'SMA_60']
    df = feature_engineering_smas(df, sma_cols)
    # df['rsi_14'] = ta.momentum.RSIIndicator(df['close'], window=14).rsi()
    # df['rsi_20'] = ta.momentum.RSIIndicator(df['close'], window=20).rsi()
    # df['rsi_30'] = ta.momentum.RSIIndicator(df['close'], window=30).rsi()

    # # Calculando el cambio relativo del RSI respecto al día anterior
    # df['rsi_change_14'] = df['rsi_14'].diff()
    # df['rsi_change_20'] = df['rsi_20'].diff()
    # df['rsi_change_30'] = df['rsi_30'].diff()

    # # Calculando la tasa de cambio porcentual del RSI
    # df['rsi_pct_change_14'] = df['rsi_14'].pct_change()
    # df['rsi_pct_change_20'] = df['rsi_20'].pct_change()
    # df['rsi_pct_change_30'] = df['rsi_30'].pct_change()

    # # Calculando la tendencia del RSI usando una media móvil simple de los cambios
    # df['rsi_trend_14'] = df['rsi_change_14'].rolling(window=5).mean()
    # df['rsi_trend_20'] = df['rsi_change_20'].rolling(window=5).mean()
    # df['rsi_trend_30'] = df['rsi_change_30'].rolling(window=5).mean()

    # Limpiar datos para remover NaNs generados por las operaciones rolling y diff
    df.dropna(inplace=True)

    # Moving Average Convergence Divergence (MACD)
    # macd = ta.trend.MACD(df['close'])
    # df['macd_line'] = macd.macd()
    # df['macd_signal'] = macd.macd_signal()
    # df['macd_diff'] = macd.macd_diff()

    # df['macd_line'] = df['macd_line']/df['macd_signal']

    # Rate of Change (ROC)
    # df['roc'] = ta.momentum.ROCIndicator(df['close'], window=12).roc()

    # Eliminar NaN
    df.dropna(inplace=True)

    return df

In [ ]:
# Stochastic Oscillator
stochastic = ta.momentum.StochasticOscillator(df_EURUSD_5min['high'], df_EURUSD_5min['low'], df_EURUSD_5min['close'], window=14, smooth_window=3)
df_EURUSD_5min['stoch'] = stochastic.stoch()
df_EURUSD_5min['stoch_signal'] = stochastic.stoch_signal()

# Average True Range (ATR)
df_EURUSD_5min['ATR'] = ta.volatility.AverageTrueRange(df_EURUSD_5min['high'], df_EURUSD_5min['low'], df_EURUSD_5min['close'], window=14).average_true_range()

# Ichimoku Kinko Hyo
ichimoku = ta.trend.IchimokuIndicator(df_EURUSD_5min['high'], df_EURUSD_5min['low'], window1=9, window2=26, window3=52)
df_EURUSD_5min['ichimoku_a'] = ichimoku.ichimoku_a()
df_EURUSD_5min['ichimoku_b'] = ichimoku.ichimoku_b()

# True Strength Index (TSI)
df_EURUSD_5min['TSI'] = ta.momentum.TSIIndicator(df_EURUSD_5min['close']).tsi()

# KAMA (Kaufman's Adaptive Moving Average)
df_EURUSD_5min['KAMA'] = ta.momentum.KAMAIndicator(df_EURUSD_5min['close']).kama()

# Percentage Price Oscillator (PPO)
df_EURUSD_5min['PPO'] = ta.momentum.PercentagePriceOscillator(df_EURUSD_5min['close']).ppo()

# StochRSI (Stochastic Relative Strength Index)
stoch_rsi = ta.momentum.StochRSIIndicator(df_EURUSD_5min['close'])
df_EURUSD_5min['stochrsi'] = stoch_rsi.stochrsi()

# Williams %R
df_EURUSD_5min['williams_r'] = ta.momentum.WilliamsRIndicator(df_EURUSD_5min['high'], df_EURUSD_5min['low'], df_EURUSD_5min['close']).williams_r()

# Aroon Indicator
aroon = ta.trend.AroonIndicator(df_EURUSD_5min['close'],df_EURUSD_5min['low'])
df_EURUSD_5min['aroon_up'] = aroon.aroon_up()
df_EURUSD_5min['aroon_down'] = aroon.aroon_down()

# Detrended Price Oscillator (DPO)
df_EURUSD_5min['DPO'] = ta.trend.DPOIndicator(df_EURUSD_5min['close']).dpo()

# Vortex Indicator (without volume)
vortex = ta.trend.VortexIndicator(df_EURUSD_5min['high'], df_EURUSD_5min['low'], df_EURUSD_5min['close'])
df_EURUSD_5min['vortex_pos'] = vortex.vortex_indicator_pos()
df_EURUSD_5min['vortex_neg'] = vortex.vortex_indicator_neg()

In [ ]:
scale_pos_weight = len(df_EURUSD_5min[df_EURUSD_5min['Valid_Trade'] == 0]) / len(df_EURUSD_5min[df_EURUSD_5min['Valid_Trade'] == 1]) 
scale_pos_weight

In [ ]:
exclude_cols = ['date', 'open', 'high', 'low', 'close', 'volume', 'barCount',
       'datetime', 'Year', 'Month', 'Day','No_Price', 'Target_Price', 'Max_Future_Close', 'Min_Future_Close',
       'Hit_Target', 'Avoid_Drop']

In [ ]:
df = df_EURUSD_5min.drop(columns=exclude_cols).dropna()

In [ ]:
#Train and Test
df_train = df[df['Date'].astype(int)<20230601]
df_test = df[df['Date'].astype(int)>=20230601]


# Dividir datos en conjuntos de entrenamiento y prueba
X_train = df_train.drop(columns=['Date','Valid_Trade'])
y_train = df_train['Valid_Trade']  

X_test = df_test.drop(columns=['Date','Valid_Trade'])
y_test = df_test['Valid_Trade']  


In [ ]:
# Setup XGBoost parameters
params = {
    'objective': 'binary:logistic',
    'learning_rate': 0.1,
    'n_estimators': 100,
    'scale_pos_weight': scale_pos_weight,
    'use_label_encoder': False,
    'eval_metric': 'logloss'
}

# Initialize and train the XGBoost classifier
model = xgb.XGBClassifier(**params)
model.fit(X_train, y_train)

In [ ]:

# Evaluar el modelo
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))


In [ ]:
probabilities = model.predict_proba(X_test)[:, 1]
# Assume y_test is your true labels
results_df = pd.DataFrame({
    'PredictedProbabilities': probabilities,
    'TrueLabel': y_test
})

# Optional: Reset index if y_test is not aligned by index
results_df.reset_index(drop=True, inplace=True)


In [ ]:
bins = [0, 0.5, 0.7, 0.8, 0.9, 0.98, 1.0]
labels = ['< 50%', '50-70%', '70-80%', '80-90%', '90-98%', '> 98%']

results_df['ProbabilityRange'] = pd.cut(results_df['PredictedProbabilities'], bins=bins, labels=labels, include_lowest=True)

In [ ]:
# Group by the ProbabilityRange and calculate counts and sum of TrueLabel
summary_df = results_df.groupby('ProbabilityRange').agg(
    Occurrences=('PredictedProbabilities', 'count'),  # Count occurrences in each bin
    SumTrueLabels=('TrueLabel', 'sum')  # Sum of TrueLabels to find how many were actually positive
).reset_index()

# Calculate precision for each range
summary_df['Precision'] = summary_df['SumTrueLabels'] / summary_df['Occurrences']

# Calculate total sum of true labels for recall calculation
total_true_labels = summary_df['SumTrueLabels'].sum()

# Calculate recall for each range
summary_df['Recall'] = summary_df['SumTrueLabels'] / total_true_labels

summary_df

In [ ]:
data_viz = df_EURUSD_5min[df_EURUSD_5min['Date'].astype(int)>=20240301]
data_viz = data_viz.set_index('datetime')

fig = go.Figure(data=[go.Candlestick(x=data_viz.index,
                open=data_viz['open'],
                high=data_viz['high'],
                low=data_viz['low'],
                close=data_viz['close'])])

# Adding support and resistance lines
# for level in resistance_array_h4:
#     fig.add_trace(go.Scatter(x=data_viz.index, y=[level[0]]*len(data_viz), mode='lines', name='Resistance',
#                              line=dict(color='green', width=level[1], dash='longdash')))

# for level in support_array_h4:
#     fig.add_trace(go.Scatter(x=data_viz.index, y=[level[0]]*len(data_viz), mode='lines', name='Support',
#                              line=dict(color='red', width=level[1], dash='longdash')))
    
#Adding support and resistance lines
for level in resistance_array_5min:
    fig.add_trace(go.Scatter(x=data_viz.index, y=[level[0]]*len(data_viz), mode='lines', name='Resistance 5',
                             line=dict(color='green', width=0.5, dash='dot')))

for level in support_array_5min:
    fig.add_trace(go.Scatter(x=data_viz.index, y=[level[0]]*len(data_viz), mode='lines', name='Support 5',
                             line=dict(color='red', width=0.5, dash='dot')))


# Update layout
fig.update_layout(title='Interactive EUR/USD Candlestick Chart',
                  xaxis_title='Date',
                  yaxis_title='Price (EUR/USD)',
                  yaxis=dict(
                        tickmode='auto',  # 'auto' lets Plotly decide the placement of ticks
                        tickformat=".4f",  # 4f for four decimal places
                    ),
                  xaxis_rangeslider_visible=False)  # Disable range slider

fig.show()